In [1]:
#Теплотехнический расчет ограждающей конструкции жилого дома.
P_o=[1000,1800,1600,80]
sigma=[0.12,0.38,0.02,None]
alfa=[0.76,0.07,0.33,0.35]

#karakol
Zот=187
tb=-1.1
tиC=-12
a,b,r=0.00035,1.4,0.87



In [2]:
from dataclasses import dataclass

@dataclass
class Mytest:
    P_o:float
    sigma:float
    alfa:float
    def show(self):
        return [self.P_o,self.sigma,self.alfa]


tabl=[Mytest(P_o[i],sigma[i],alfa[i]) for i in (0,1,2,3)]
print(tabl[0])

Mytest(P_o=1000, sigma=0.12, alfa=0.76)


In [3]:
for i in tabl[0].show():
    print(i)
#Определение требуемого Rreq термического сопротивления теплопередаче ОКи толщины слоя утеплителя
#Определяем градусо-сутки отопительного периода по формуле
print("D=",D:=(tb-tиC)*Zот)
#Определяем нормированное сопротивление теплопередаче по формуле
print("Rreq=",Rreq:=a*D+b)
#Находим требуемое условное сопротивление теплопередаче по формуле
print("R_o_уст=",R_o_уст:=Rreq/r)
#Требуемое значение сопротивления теплопередаче слоя утеплителя из плит минераловатных
#прошивных на синтетическом связующем находим согласно п. 8 СНиП КР 23-01:2013 по формуле
print("Rут=",Rут:=R_o_уст-((Rb:=1/(a_в:=8.7))+
(R_h:=1/(a_h:=23))+
(SumR:=sum([tabl[i].sigma/tabl[i].alfa 
for i in range(3)]))))
#Расчетную толщину утеплителя находим по формуле
print("sigma_ут=",sigma_ут:=Rут*tabl[3].alfa)
tabl[3].sigma,tabl[3].sigma=[sigma_ут for i in (0,1)]
#Для проверки полученного результата находим приведенное сопротивление теплопередаче
#наружной стены по формуле
print("R_0=",R_0:=(1/a_в+1/a_h+r*sum(
        [tabl[i].sigma/tabl[i].alfa for i in range(4)]
)))

#Проверяем выполнение неравенства (достаточность сопротивления теплопередаче):
print(round(R_0,12)>=round(Rreq,12),f"{R_0}>{Rreq}")

1000
0.12
0.76
D= 2038.3
Rreq= 2.1134049999999998
R_o_уст= 2.429201149425287
Rут= -3.3762918661995043
sigma_ут= -1.1817021531698264
R_0= 2.133999702648675
True 2.133999702648675>2.1134049999999998


In [5]:
#Расчет потерь теплоты через ограждающие конструкции
class my():
    def __init__(
        self,
        name:str,
        side:str,
        size,
        Rreq:float,
        temp_diff:list[float,float]#temperate difference
    ):
        self.name=name
        self.side=side
        self.size=list(size),
        self.square=round(sum(i/100 for i in size),3)
        self.K=round(1/Rreq,3)
        self.n=1
        self.temp_diff=round((temp_diff[0]-temp_diff[1])*self.n,3)
        self.Q=round(self.square*self.K*self.temp_diff,3) #main_t_d
        self.beta1=0.1 if side=="n" else 0.05
        self.beta2=0.05
        self.res=round(1+self.beta1+self.beta2 ,3)
        self.Qogr=round(self.Q*self.res,3)#heat_loss
    def to_vec(self):
        return [list("name side size square K n temp_diff Q beta1 beta2 res Qogr".split(" ")),
        [self.name,self.side,self.size,self.square,self.K,self.n,
         self.temp_diff,self.Q,self.beta1,self.beta2,self.res,self.Qogr]]
        
table=[my("несущяя стена","n",[2604,2700],Rreq,[18,-20]),
my("несущяя стена","w",[4797,2700],Rreq,[18,-20]),
my("окно","w",[1500,1500],Rreq,[18,-20]),
my("несущяя стена","w",[4198,2700],Rreq,[19,-20]),
my("окно","w",[750,1500],Rreq,[19,-20]),
my("окно","w",[750,1500],Rreq,[19,-20]),
my("несущяя стена","s",[7208,2700],Rreq,[19,-20]),
my("окно","s",[750,1500],Rreq,[19,-20]),
my("окно","s",[750,1500],Rreq,[19,-20]),
my("окно","s",[1500,1500],Rreq,[19,-20]),
my("несущяя стена","w",[5130,2700],Rreq,[19,-20]),
my("несущяя стена","w",[3386,2700],Rreq,[19,-20]),
my("окно","s",[1500,1500],Rreq,[19,-20]),
my("окно","e",[1500,1500],Rreq,[19,-20]),
my("несущяя стена","e",[2046,2700],Rreq,[15,-17]),
my("окно","e",[750,1500],Rreq,[15,-17]),
my("несущяя стена","e",[3363,2700],Rreq,[17,-19]),
my("дверь","e",[900,2100],Rreq,[17,-19]),
my("несущяя стена","n",[1957,2700],Rreq,[17,-19]),
my("несущяя стена","n",[2602,2700],Rreq,[15,-17]),
my("окно","n",[750,1500],Rreq,[15,-17]),
my("несущяя стена","n",[2338,2700],Rreq,[15,-17]),
my("несущяя стена","n",[1766,2700],Rreq,[18,-20])]
ntabl=[["Название","ориентация","Размер","площ.","Теплопередача К ВТ(м**°С","Положения n","Разность температур(tb-tн)*n",
        "Основные теплопотери Qосн, Вт","На ориентацию β1","Прочая β2","1+Σβ","Теплопотери Qогр,Вт"]]

for show in table:
    ntabl+=[show.to_vec()[1]]
    #print(show.to_vec()[1])
#print(ntabl)
import pandas as pd
pd.DataFrame(ntabl)

,0,1,2,3,4,5,6,7,8,9,10,11
0,Название,ориентация,Размер,площ.,Теплопередача К ВТ(м**°С,Положения n,Разность температур(tb-tн)*n,"Основные теплопотери Qосн, Вт",На ориентацию β1,Прочая β2,1+Σβ,"Теплопотери Qогр,Вт"
1,несущяя стена,n,"([2604, 2700],)",53.04,0.473,1,38,953.341,0.1,0.05,1.15,1096.342
2,несущяя стена,w,"([4797, 2700],)",74.97,0.473,1,38,1347.511,0.05,0.05,1.1,1482.262
3,окно,w,"([1500, 1500],)",30.0,0.473,1,38,539.22,0.05,0.05,1.1,593.142
4,несущяя стена,w,"([4198, 2700],)",68.98,0.473,1,39,1272.474,0.05,0.05,1.1,1399.721
5,окно,w,"([750, 1500],)",22.5,0.473,1,39,415.058,0.05,0.05,1.1,456.564
6,окно,w,"([750, 1500],)",22.5,0.473,1,39,415.058,0.05,0.05,1.1,456.564
7,несущяя стена,s,"([7208, 2700],)",99.08,0.473,1,39,1827.729,0.05,0.05,1.1,2010.502
8,окно,s,"([750, 1500],)",22.5,0.473,1,39,415.058,0.05,0.05,1.1,456.564
9,окно,s,"([750, 1500],)",22.5,0.473,1,39,415.058,0.05,0.05,1.1,456.564
